# Track D / Day 4 — Distribution-match validation (Colab)

Implements `pilot_0_1_execution_spec.md` §2.2 step 4 — the three **mandatory, pre-registered** acceptance tests (distinct from Day 2's own informal length check). All compare the Day-3-surviving ghost answers against TOFU's real **holdout10** split:
1. Token length — KS test / Cohen's d (same tokenizer as Day 1/2).
2. Perplexity under the **base** `meta-llama/Llama-3.2-1B-Instruct` (not the TOFU-tuned model) — same criteria.
3. SBERT centroid cosine distance, ghost↔holdout ≤ 1.25 × holdout↔forget10 (`DECISIONS.md` item 3).

**Before running:**
- **Switch this notebook's runtime to GPU** (Runtime → Change runtime type → T4 GPU) — test 2 runs a 1B-parameter model over ~800 texts; it works on CPU but will be much slower.
- Add a Colab secret `HF_TOKEN` — a Hugging Face access token (read-only is enough) from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). Your HF account also needs **gated access already approved** for `meta-llama/Llama-3.2-1B-Instruct` (request it on the model's page if you haven't).

No paid API calls anywhere in this script — HF model downloads and TOFU/Wikidata data are all free.

## 1. Mount Drive and pull the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/unlearning_pilot'
REPO_DIR = os.path.join(PROJECT_DIR, 'unlearning-audit-study')
os.makedirs(PROJECT_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/shravanidhus31/unlearning-audit-study.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log -1 --oneline

## 2. Confirm required files are present

In [ ]:
import os
for p in ['scripts/day4_validation.py', 'ghosts/candidates_filtered.jsonl']:
    assert os.path.exists(p), f'{p} not found -- re-run cell 1, or check Day 3 has been committed.'
print('All required files present.')

## 3. Install dependencies

In [ ]:
!pip install -q transformers datasets scipy sentence-transformers huggingface_hub torch

## 4. Secrets (HF token for the gated base model)

In [ ]:
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('HF_TOKEN loaded:', bool(os.environ.get('HF_TOKEN')))

## 5. Self-test (offline — no network/GPU, sanity-check before the real run)

In [ ]:
!python scripts/day4_validation.py --selftest

## 6. Full run
Runs all 3 tests against `ghosts/candidates_filtered.jsonl` (the 420 rows that survived Day 3). Test 2 (perplexity) is the slow one — downloads the base 1B model once, then runs it over ~420 ghost + ~400 holdout10 answers.

In [ ]:
!python scripts/day4_validation.py --outdir ghosts

## 7. Read the results

In [ ]:
with open('ghosts/validation_report.md', encoding='utf-8') as f:
    print(f.read())

## 8. Commit the results manually
Review `ghosts/validation_report.md` yourself first, then run the commands below (this notebook does not push automatically).

In [ ]:
print('To commit manually:')
print('  git -C', REPO_DIR, 'add ghosts/validation_report.md ghosts/validation_results.json ghosts/sbert_checkpoint_pin.json')
print('  git -C', REPO_DIR, 'commit -m "Track D Day 4: validation battery on Day 3 survivors"')
print('  git -C', REPO_DIR, 'push')